In [1]:
import re, requests, subprocess
import pandas as pd
import numpy as np
from pathlib import Path
from time import sleep
from concurrent.futures import ThreadPoolExecutor, as_completed
from astropy.io import fits as astrofits
import glob

CATALOG_PATH = "cumulative_2026_04_10_20_25_58.csv"
FITS_DIR     = Path("./downloaded-koi-lcs/")
OUTPUT_PKL   = "koi_cumulative_lcs.pkl"
BATCH_SIZE   = 200   # larger batches = fewer scraping rounds
N_SCRAPE_WORKERS = 5  # parallel URL discovery
N_CURL_WORKERS   = 5  # parallel downloads

FITS_DIR.mkdir(exist_ok=True)

In [2]:
def get_kepler_fits_urls(kic):
    kic9 = f"{int(kic):09d}"
    url  = f"https://archive.stsci.edu/pub/kepler/lightcurves/{kic9[:4]}/{kic9}/"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        files = re.findall(r'href="([^"]+_llc\.fits)"', r.text)
        return [url + f for f in files]
    except:
        return []

In [3]:
def scrape_urls_parallel(kic_batch):
    """Scrape all URLs for a batch of KICs in parallel."""
    all_urls = []
    with ThreadPoolExecutor(max_workers=N_SCRAPE_WORKERS) as ex:
        futures = {ex.submit(get_kepler_fits_urls, kic): kic for kic in kic_batch}
        for future in as_completed(futures):
            all_urls.extend(future.result())
    return all_urls

In [4]:
def download_parallel(urls, fits_dir, n_workers=5):
    url_file = fits_dir / "urls.txt"  # write urls.txt INSIDE fits_dir
    url_file.write_text("\n".join(urls) + "\n")
    subprocess.run(
        f"xargs -P {n_workers} -I{{}} curl -O --silent {{}} < urls.txt",
        shell=True, cwd=fits_dir, check=True
    )
    url_file.unlink()

In [5]:
def extract_batch(fits_dir):
    records = []
    for fpath in Path(fits_dir).glob("*_llc.fits"):  # use Path.glob, not glob.glob
        try:
            with astrofits.open(fpath) as hdul:
                data    = hdul[1].data
                kic     = hdul[0].header.get('KEPLERID')
                quarter = hdul[0].header.get('QUARTER')
                mask    = np.isfinite(data['TIME']) & np.isfinite(data['PDCSAP_FLUX'])
                records.append({
                    "KIC":         kic,
                    "quarter":     quarter,
                    "time":        data['TIME'][mask],
                    "detflux":     data['PDCSAP_FLUX'][mask],
                    "detflux_err": data['PDCSAP_FLUX_ERR'][mask],
                    "n_points":    int(mask.sum())
                })
        except Exception as e:
            print(f"Error: {fpath}: {e}")
    
    print(f"  Extracted {len(records)} records from {sum(1 for _ in Path(fits_dir).glob('*_llc.fits'))} files")
    return records

In [6]:
# Load + deduplicate
catalog     = pd.read_csv("cumulative_koi.csv", comment="#")
unique_kics = catalog['kepid'].dropna().astype(int).drop_duplicates().tolist()

# Resume support
if Path(OUTPUT_PKL).exists():
    done_kics   = set(pd.read_pickle(OUTPUT_PKL)['KIC'].tolist())
    all_records = pd.read_pickle(OUTPUT_PKL).to_dict("records")
else:
    done_kics   = set()
    all_records = []

remaining = [k for k in unique_kics if k not in done_kics]
batches   = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
print(f"{len(remaining)} KICs remaining across {len(batches)} batches")

8214 KICs remaining across 42 batches


In [7]:
for batch_num, batch in enumerate(batches):
    print(f"\n--- Batch {batch_num+1}/{len(batches)} ---")

    # 1. Scrape URLs in parallel
    print("Discovering URLs...")
    urls = scrape_urls_parallel(batch)
    print(f"  Found {len(urls)} files")

    # 2. Download
    print("Downloading...")
    download_parallel(urls, FITS_DIR, n_workers=N_CURL_WORKERS)

    # 3. Extract
    print("Extracting...")
    fits_count = sum(1 for _ in FITS_DIR.glob("*_llc.fits"))
    print(f"  FITS files on disk: {fits_count}")
    batch_records = extract_batch(FITS_DIR)
    print(f"  Records extracted: {len(batch_records)}")

    # 4. Only save + delete if extraction worked
    if len(batch_records) > 0:
        all_records.extend(batch_records)
        pd.DataFrame(all_records).to_pickle(OUTPUT_PKL)
        print(f"  Pickle saved — {len(all_records)} records total")
        for f in FITS_DIR.glob("*_llc.fits"):
            f.unlink()
    else:
        print("  WARNING: 0 records extracted, keeping FITS files for inspection")

    sleep(2)

print("\nAll done!")


--- Batch 1/42 ---
Discovering URLs...
  Found 3214 files
Downloading...
Extracting...
  FITS files on disk: 3214
  Extracted 3214 records from 3214 files
  Records extracted: 3214
  Pickle saved — 3214 records total

--- Batch 2/42 ---
Discovering URLs...
  Found 3243 files
Downloading...
Extracting...
  FITS files on disk: 3243
  Extracted 3243 records from 3243 files
  Records extracted: 3243
  Pickle saved — 6457 records total

--- Batch 3/42 ---
Discovering URLs...
  Found 3226 files
Downloading...
Extracting...
  FITS files on disk: 3226
  Extracted 3226 records from 3226 files
  Records extracted: 3226
  Pickle saved — 9683 records total

--- Batch 4/42 ---
Discovering URLs...
  Found 3238 files
Downloading...
Extracting...
  FITS files on disk: 3238
  Extracted 3238 records from 3238 files
  Records extracted: 3238
  Pickle saved — 12921 records total

--- Batch 5/42 ---
Discovering URLs...
  Found 3134 files
Downloading...
Extracting...
  FITS files on disk: 3134
  Extracted 

CalledProcessError: Command 'xargs -P 5 -I{} curl -O --silent {} < urls.txt' returned non-zero exit status 1.